# GPU implementation

$\renewcommand{\ket}[1]{\left|#1\right\rangle}\renewcommand{\bra}[1]{\left\langle #1\right|}\renewcommand{\braket}[2]{\left\langle #1 \middle| #2 \right\rangle}\renewcommand{\ketbra}[2]{\left|#1\right\rangle\!\left\langle #2\right|}$

This notebook explains how the GPU timing estimates for the classical MCMC algorithm are obtained.

**Table of contents**

1. [Source](#source)
2. [Algorithm](#algorithm)
3. [Timing calculation](#timing-calculation)
4. [Reproducibility](#reproducibility)

<a id="source"></a>
## Source

The code and data are available in the `estimation_timing/gpu` folder.

The folder contains:

* `timing_estimation_gpu.cu`: CUDA/C++ source code containing the SK instance generation, GPU memory wrappers, random-number pools, chain-state storage, local and uniform Metropolis kernels, and CUDA-event timing logic.
* `timing_estimation_gpu.slurm`: SLURM script that compiles `timing_estimation_gpu.cu` for different values of `SK_N` and runs the resulting GPU benchmarks. The script contains generic settings; the precise settings used for the reported runs are described below.
* `h100/test<i>`: directories containing identical runs performed at different times on the same device class. Each directory contains a copy of the source code, the SLURM script used for execution, and logs reporting the hardware characteristics, compiler configuration, and timing results.

<a id="algorithm"></a>
## Algorithm

The benchmark compares the local single-spin-flip and uniform proposal moves. Given a current configuration $x$ and a proposed configuration $y$, the proposal is accepted with probability

$$
A_{yx}^{(\beta)}
=
\min\left\{
1,
\exp\left[-\beta\left(H(y)-H(x)\right)\right]
\right\}.
$$

The local proposal computes

$$
\Delta_i H
=
2x_i
\left(
\widetilde h_i+\sum_j\widetilde J_{ij}x_j
\right),
$$

and therefore requires $O(n)$ arithmetic operations per attempted proposal. The uniform proposal samples a complete spin configuration and evaluates its dense SK energy,

$$
H(y)
=
-\sum_i\widetilde h_i y_i
-\sum_{i<j}\widetilde J_{ij}y_i y_j,
$$

requiring $O(n^2)$ arithmetic operations per attempted proposal.

The GPU implementation assigns one CUDA block to each Markov chain, with `BLOCK_THREADS = 128` threads per block. The chain evolution remains sequential in Markov-chain time. GPU parallelism is therefore used only to accelerate the arithmetic within each proposal. Ignoring synchronization and reduction overheads, the arithmetic work per proposal is distributed as $O(n/T)$ for the local move and $O(n^2/T)$ for the uniform move, where $T$ is the number of threads in the block.

The implementation uses single-precision floating-point arithmetic for the fields, couplings, energies, and Metropolis probabilities.

### Common GPU utilities

In CUDA, a `__global__` function is a kernel launched by the CPU and executed on the GPU by many threads. A `__shared__` variable is stored in memory local to one CUDA block. All threads in the same block can access it, while different blocks have independent copies. Here, shared memory is used for block-level reductions, the trial state of the uniform proposal, and scalar variables shared by the threads evolving one chain.

**Constants**

* `SRC`: CUDA/C++ source file compiled by the SLURM script.
* `NS`: list of system sizes, here $n\in\{64,128,256,512\}$.
* `SK_N`: number of spins, passed at compile time as `-DSK_N=<n>` for each value in `NS`.
* `N_MODELS`: number of independent SK instances generated in one benchmark execution.
* `CHAINS_PER_MODEL`: number of Markov chains simulated for each SK instance.
* `N_STEPS`: number of attempted Metropolis proposals per chain.
* `SEED`: seed used for instance generation and random-pool construction.
* `BETA`: inverse temperature used in the Metropolis acceptance probability.
* `BLOCK_THREADS`: number of CUDA threads assigned to each chain.
* `RNG_POOL_SIZE`: size of the precomputed random-number pools.
* `GPU_ARCH`: CUDA architecture passed to `nvcc`. The supplied script uses `sm_80`; the reported run was executed on an NVIDIA H100 NVL GPU.
* `N_CHAINS`: total number of chains, defined as `N_MODELS * CHAINS_PER_MODEL`.

**Helper classes and methods**

* `CUDA_CHECK`: macro that wraps CUDA runtime calls. If a call fails, it prints the CUDA error and terminates the program.
* `DeviceArray<T>`: owner of a GPU allocation created with `cudaMalloc` and released with `cudaFree`. Copying is disabled to prevent double ownership. The `data()` methods expose the device pointer, while `copy_from(v)` transfers a host `std::vector<T>` to the device.
* `GpuTimer`: CUDA-event timer used to measure kernel runtime. The constructor creates two CUDA events. The `start()` method records the first event, while `stop_seconds()` records and synchronizes the second event, evaluates the elapsed time with `cudaEventElapsedTime`, and returns it in seconds. The measured interval therefore contains only GPU kernel execution.
* `RandomPool`: host-side owner of the random-number pools. Before the timed region, it generates `h_int`, containing random `uint64_t` values, and `h_float`, containing uniformly distributed single-precision values in $[0,1)$. These arrays are copied to the device as `d_int` and `d_float`.
* `RandomPoolView`: device-side view of the random-number pools. The methods `randint(chain, step, tag)` and `uniform(chain, step, tag)` return a precomputed integer or uniform value. The method `spin(chain, step, i)` extracts a random spin in $\{-1,+1\}$. The pool index is a deterministic function of the chain, Markov-chain step, and tag, reduced modulo `RNG_POOL_SIZE`. The random values are therefore generated once and reused cyclically during the timed kernel. This isolates the arithmetic and memory costs of the Metropolis kernels and does not constitute a production-quality random-number scheme.
* `ModelBank`: host-side owner of the SK instances. It stores all fields in `h_host` and the dense coupling matrices in `J_host`, generates normalized SK instances in `init_models(seed)`, and copies them to `h_dev` and `J_dev`. The method `energy(model, s)` evaluates the dense SK energy on the host and is used to initialize each chain energy.
* `ModelBankView`: device-side view of the model bank. The methods `h_model(model)` and `J_model(model)` return pointers to the field vector and dense $n\times n$ coupling matrix of one instance.
* `ChainState`: host-side owner of the device chain state. It allocates `s_dev` for all spin configurations and `E_dev` for the current energy of each chain. It also computes `E0_host`, the energy of the all-plus initial configuration for each chain. The method `reset_energy()` copies these values back to the GPU before timing each kernel.
* `ChainStateView`: device-side view of the chain state. The method `state(chain)` returns the spin array for one chain, `energy(chain)` returns a reference to its current energy, `flip(chain, i)` flips one spin, and `reset_state(chain)` resets the chain to the all-plus configuration.
* `IsingEnergyBuffer`: shared-memory scratch space allocated once per block. It contains `red[BLOCK_THREADS]`, used for reductions, and `trial[SK_N]`, used to store the uniformly proposed configuration. The method `block_sum(x)` performs a binary-tree reduction. The method `local_delta_energy(h, J, s, i)` evaluates $\Delta_iH$ by distributing the row sum across the block. The methods `assign_random_state_to_trial`, `get_energy_trial_state`, and `accept_trial_state` generate, evaluate, and conditionally store a uniformly proposed configuration.
* `metropolis_accept`: device-side implementation of the Metropolis acceptance rule. It accepts automatically when $\Delta H\leq0$ and otherwise compares a precomputed uniform random value with `__expf(-BETA * dE)`. The use of `__expf` is consistent with the `--use_fast_math` compiler option.
* `reset_states`: CUDA kernel that resets every chain to the all-plus configuration. It is executed before each proposal benchmark and outside the timed region.

### Local move

The local move is implemented by the `local_kernel` CUDA kernel, launched with one block per Markov chain and `BLOCK_THREADS` threads per block. Thus, `blockIdx.x` identifies the chain and `threadIdx.x` identifies a worker thread within that chain.

At each Markov-chain step, thread `0` samples the spin index $i$ using `rng.randint`. The index is stored in `__shared__ int i`, allowing all threads in the block to access it after synchronization. The block then calls `energy_helper.local_delta_energy(h, J, s, i)`. Each thread evaluates part of the row sum $\sum_j\widetilde J_{ij}x_j$, and the partial results are combined by `block_sum`.

Thread `0` performs the Metropolis acceptance test. When the move is accepted, it flips spin $i$ in the persistent chain state and updates the stored energy by adding $\Delta_iH$. A synchronization at the end of each step ensures that all threads observe the updated state before the next proposal.

### Uniform move

The uniform move is implemented by the `uniform_kernel` CUDA kernel, again using one block per Markov chain.

At each Markov-chain step, all threads call `assign_random_state_to_trial`, which fills the shared-memory array `trial[SK_N]` with a new spin configuration. The proposed configuration remains in shared memory until the Metropolis decision is known. The block then calls `get_energy_trial_state(h, J)`, which distributes the dense energy calculation across its threads. Each thread accumulates part of the field and interaction terms, and `block_sum` combines the partial results.

Thread `0` computes $\Delta H=H(y)-H(x)$ and applies the Metropolis rule. When the proposal is accepted, it updates the stored energy, and all threads cooperatively copy the shared trial configuration into the persistent chain state through `accept_trial_state`.


<a id="timing-calculation"></a>
## Timing calculation

CUDA events measure the proposal kernels only. Memory allocation, host-to-device transfers, random-pool construction, state initialization, and SK instance generation are outside the timed region.

The script compiles `SRC = timing_estimation_gpu.cu` for $n\in\{64,128,256,512\}$ using `-DSK_N=<n>`. The reported benchmark uses `N_MODELS = 1`, `CHAINS_PER_MODEL = 1`, `N_STEPS = 1000000`, `SEED = 123456789`, `BETA = 1.0f`, `BLOCK_THREADS = 128`, `RNG_POOL_SIZE = 1048576`, and `GPU_ARCH = sm_80`. The corresponding log identifies the device as an NVIDIA H100 NVL GPU.

Because `N_MODELS = 1` and `CHAINS_PER_MODEL = 1`, the benchmark evolves a single Markov chain. The reported kernel times are therefore single-chain latency measurements rather than saturated H100 throughput measurements. A throughput benchmark would require enough independent chains to keep many CUDA blocks resident simultaneously.

The program reports `local_seconds` and `uniform_seconds`. For the single-chain configuration, the average latency per attempted proposal is `seconds / N_STEPS`. More generally, for independent chains evolved concurrently, it is `seconds / (N_CHAINS * N_STEPS)`.

The tables below report three runs using the same benchmark configuration. Run 1 is reproduced by the supplied log, while Runs 2 and 3 are additional measurements from the same experiment set.

The local-move timings are the CUDA-event kernel times for `N_STEPS = 1000000` attempted proposals:

| Run | $n=64$ | $n=128$ | $n=256$ | $n=512$ |
|---|---:|---:|---:|---:|
| Run 1 | 0.928508 | 0.937455 | 1.125960 | 1.557990 |
| Run 2 | 0.926027 | 0.935282 | 1.121310 | 1.547510 |
| Run 3 | 0.926223 | 0.935248 | 1.117720 | 1.547070 |
| Average | 0.926919 | 0.935995 | 1.121663 | 1.550857 |

The uniform-move timings are the CUDA-event kernel times for `N_STEPS = 1000000` attempted proposals:

| Run | $n=64$ | $n=128$ | $n=256$ | $n=512$ |
|---|---:|---:|---:|---:|
| Run 1 | 2.050120 | 3.196240 | 8.695600 | 59.846200 |
| Run 2 | 2.051920 | 3.201940 | 8.702960 | 59.349300 |
| Run 3 | 2.051810 | 3.201820 | 8.697330 | 59.360100 |
| Average | 2.051283 | 3.200000 | 8.698630 | 59.518533 |

We fit the local latency as

$$
\tau_{\mathrm{GPU}}^{\mathrm{loc}}(n)
=
a_{\mathrm{GPU}}^{\mathrm{loc}}
+
b_{\mathrm{GPU}}^{\mathrm{loc}}n,
$$

and the uniform latency as

$$
\tau_{\mathrm{GPU}}^{\mathrm{unif}}(n)
=
a_{\mathrm{GPU}}^{\mathrm{unif}}
+
b_{\mathrm{GPU}}^{\mathrm{unif}}n
+
c_{\mathrm{GPU}}^{\mathrm{unif}}n^2.
$$

The following code fits the averaged total runtimes using non-negative least squares. The single-step coefficients are obtained by dividing the fitted total-runtime coefficients by `N_STEPS`.


In [1]:
import numpy as np
from scipy.optimize import nnls

N_STEPS = 1_000_000
ns = np.array([64, 128, 256, 512], dtype=float)

# Rows are independent runs; columns correspond to n = 64, 128, 256, 512.
local_time_gpu = np.array(
    [
        [0.928508, 0.937455, 1.125960, 1.557990],
        [0.926027, 0.935282, 1.121310, 1.547510],
        [0.926223, 0.935248, 1.117720, 1.547070],
    ]
)

uniform_time_gpu = np.array(
    [
        [2.050120, 3.196240, 8.695600, 59.846200],
        [2.051920, 3.201940, 8.702960, 59.349300],
        [2.051810, 3.201820, 8.697330, 59.360100],
    ]
)

average_local_time_gpu = np.mean(local_time_gpu, axis=0)
average_uniform_time_gpu = np.mean(uniform_time_gpu, axis=0)


def fit_nonnegative_polynomial(y: np.ndarray, degree: int) -> np.ndarray:
    """Return non-negative least-squares polynomial coefficients."""
    design_matrix = np.vstack(
        [ns**power for power in range(degree + 1)]
    ).T
    return nnls(design_matrix, y)[0]


a_local, b_local = fit_nonnegative_polynomial(
    average_local_time_gpu,
    degree=1,
)
a_uniform, b_uniform, c_uniform = fit_nonnegative_polynomial(
    average_uniform_time_gpu,
    degree=2,
)

print("GPU configuration: NVIDIA H100 NVL, sm_80")
print(f"  average local timings:   {average_local_time_gpu}")
print(f"  average uniform timings: {average_uniform_time_gpu}")
print(f"  local total:   {a_local:.3e} + {b_local:.3e} n")
print(
    "  uniform total: "
    f"{a_uniform:.3e} + {b_uniform:.3e} n "
    f"+ {c_uniform:.3e} n^2"
)
print(
    "  local/step:    "
    f"{a_local / N_STEPS:.3e} "
    f"+ {b_local / N_STEPS:.3e} n"
)
print(
    "  uniform/step:  "
    f"{a_uniform / N_STEPS:.3e} "
    f"+ {b_uniform / N_STEPS:.3e} n "
    f"+ {c_uniform / N_STEPS:.3e} n^2"
)


GPU configuration: NVIDIA H100 NVL, sm_80
  average local timings:   [0.92691933 0.935995   1.12166333 1.55085667]
  average uniform timings: [ 2.05128333  3.2         8.69863    59.51853333]
  local total:   7.837e-01 + 1.459e-03 n
  uniform total: 0.000e+00 + 0.000e+00 n + 2.215e-04 n^2
  local/step:    7.837e-07 + 1.459e-09 n
  uniform/step:  0.000e+00 + 0.000e+00 n + 2.215e-10 n^2


The fitted single-step latency models used in the paper are

$$
\tau_{\mathrm{GPU}}^{\mathrm{loc}}(n)
=
\left(
7.837\times10^{-7}
+
1.459\times10^{-9}n
\right)\,\mathrm{s},
$$

and

$$
\tau_{\mathrm{GPU}}^{\mathrm{unif}}(n)
=
2.215\times10^{-10}n^2\,\mathrm{s}.
$$

Under the non-negative least-squares constraint, the fitted constant and linear terms for the uniform proposal are zero.

<a id="reproducibility"></a>
## Reproducibility

The benchmark is controlled by `timing_estimation_gpu.slurm`. The script in the main folder loads GCC 12.2.0 and attempts to load CUDA 12.1, falling back to the default CUDA module when CUDA 12.1 is unavailable. The supplied H100 log was generated on a different system using CUDA compilation tools 13.1.115.

The benchmark was executed on an NVIDIA H100 NVL GPU. The supplied script targets `GPU_ARCH = sm_80` and uses the following principal compilation flags:

```text
-std=c++17 -O3 --use_fast_math -arch=sm_80 -lineinfo -Xcompiler=-O3 -Xcompiler=-DNDEBUG
```

The Metropolis exponential is evaluated with the fast single-precision intrinsic `__expf`, consistently with `--use_fast_math`.

Run the benchmark using:

```bash
sbatch timing_estimation_gpu.slurm
```

The script compiles one executable for each value $n\in\{64,128,256,512\}$ by setting `SK_N` at compile time. The reported measurements use one SK instance, one Markov chain, and `N_STEPS = 1000000` attempted proposals. When changing these parameters, update the normalization in the fitting cell accordingly.
